# 04 · Comparação de formulações para o risco municipal

Este notebook compara duas formas de estimar o risco de um município não atingir a meta:

1. **classificação direta** de `atingiu_meta_t`;
2. **regressão de `indicador_t`**, seguida da comparação entre o nível previsto e `meta_t`.

A comparação usa o penúltimo ano observado para treino e validação agrupada por município, e o último ano observado como teste temporal. Os textos de conclusão não contêm números fixos: todas as afirmações são produzidas a partir dos resultados deste projeto.

Para manter o custo controlado no WSL, a busca padrão usa poucas combinações e uma única thread.


In [1]:
from pathlib import Path
import os
import sys
import json
import gc

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.base import clone
from sklearn.metrics import roc_auc_score

SEED = 42

def localizar_raiz(inicio: Path | None = None) -> Path:
    atual=(inicio or Path.cwd()).resolve()
    for candidato in (atual,*atual.parents):
        if (candidato/"data"/"processed").exists() and (candidato/"src").exists():
            return candidato
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto.")

PROJECT_ROOT=localizar_raiz()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0,str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

PROCESSED=PROJECT_ROOT/"data"/"processed"
MODELS=PROJECT_ROOT/"models"
REPORTS=PROJECT_ROOT/"reports"
IMAGES=PROJECT_ROOT/"images"
REPORTS.mkdir(exist_ok=True)
IMAGES.mkdir(exist_ok=True)

from src.modeling.model_registry import classificadores, regressores
from src.modeling.train_models import (
    construir_pipeline, separar_features, buscar_hiperparametros, resumo_busca,
    EXCLUIR_MUNICIPIO, TARGET_MUNICIPIO_REGRESSAO, TARGET_MUNICIPIO_CLASSIFICACAO,
)
from src.evaluation.classification_metrics import (
    avaliar_classificacao, avaliar_regressao, risco_de_nao_atingir,
    matriz_confusao, limiar_por_recall_negativos, auc_da_ausencia,
)
from src.evaluation.model_validation import (
    validar_por_grupos, validar_regressao_por_grupos,
    diferenca_bootstrap_pareada, predicao_por_uf_excluida,
)
from src.visualization.plot_style import (
    aplicar_estilo, salvar_figura, formatar_percentual,
    AZUL, LARANJA, CINZA, VERDE, VERMELHO, CORES_REGIAO,
)
from src.visualization.business_labels import rotulo

aplicar_estilo()
pd.set_option("display.max_columns",100)
pd.set_option("display.width",180)

ARQ=PROCESSED/"base_analitica_municipio.parquet"
abt=pd.read_parquet(ARQ)
observado=abt[abt["indicador_t"].notna()].copy()
observado["ano_alvo"]=pd.to_numeric(observado["ano_alvo"],errors="coerce").astype(int)
anos=sorted(observado["ano_alvo"].unique())
if len(anos)<2:
    raise ValueError("São necessários pelo menos dois anos com indicador_t observado.")
ANO_TREINO=anos[-2]
ANO_TESTE=anos[-1]
treino=observado[observado["ano_alvo"]==ANO_TREINO].copy()
teste=observado[observado["ano_alvo"]==ANO_TESTE].copy()
print(f"Treino: {ANO_TREINO} ({len(treino):,}) | teste: {ANO_TESTE} ({len(teste):,})")


Treino: 2024 (5,352) | teste: 2025 (5,485)


## 1. Preparação dos dados e auditoria das features


In [2]:
# O mesmo conjunto de features é usado nas duas formulações.
x_tr,y_tr_clf=separar_features(treino,"atingiu_meta_t",EXCLUIR_MUNICIPIO)
features=list(x_tr.columns)
x_te,y_te_clf=separar_features(teste,"atingiu_meta_t",EXCLUIR_MUNICIPIO,colunas_features=features)
y_tr_nivel=treino.loc[x_tr.index,"indicador_t"].astype(float)
y_te_nivel=teste.loc[x_te.index,"indicador_t"].astype(float)
g_tr=treino.loc[x_tr.index,"id_municipio"].to_numpy()
print(f"Features utilizadas: {len(features)}")
print(features)
print(f"Taxa de atingimento: treino={100*y_tr_clf.astype(float).mean():.1f}% | teste={100*y_te_clf.astype(float).mean():.1f}%")


[features] Removendo colunas totalmente nulas: ['indicador_t2', 'variacao_t1', 'meta_t1', 'gap_meta_t1', 'ideb_ideb_meta_ultimo']
Features utilizadas: 49
['ano_alvo', 'rede', 'sigla_uf', 'id_uf', 'nome_uf', 'nome_regiao', 'indicador_t1', 'nivel_t1', 'participacao_t1', 'meta_t', 'salto_necessario', 'ref_ideb', 'ideb_ideb_observado_ultimo', 'ideb_ideb_nota_portugues_ultimo', 'ideb_ideb_nota_matematica_ultimo', 'ideb_ideb_nota_media_ultimo', 'ideb_ideb_rendimento_ultimo', 'ideb_aprovacao_ai_pct_ultimo', 'ideb_aprovacao_1ano_pct_ultimo', 'ideb_aprovacao_2ano_pct_ultimo', 'ideb_ideb_observado_anterior', 'ideb_ideb_observado_variacao', 'area_km2', 'densidade_hab_km2', 'populacao', 'ano_referencia', 'pib_mil_reais', 'vab_adm_publica_mil_reais', 'vab_agro_mil_reais', 'vab_industria_mil_reais', 'vab_servicos_mil_reais', 'ano_pib', 'qtd_alunos_inse', 'media_inse', 'ano_referencia_inse', 'pc_nivel_1', 'pc_nivel_2', 'pc_nivel_3', 'pc_nivel_4', 'pc_nivel_5', 'pc_nivel_6', 'pc_nivel_7', 'pc_nivel_8'

## 2. Formulação A: classificação direta


In [3]:
CONFIG_CLASSIFICADORES=["logistic_regression","random_forest"]
N_ITER_CLASSIFICACAO=2
resultados_a={}
for nome in CONFIG_CLASSIFICADORES:
    estimador,distribuicoes=classificadores(n_jobs=1)[nome]
    pipeline=construir_pipeline(x_tr,estimador)
    busca=buscar_hiperparametros(
        pipeline,distribuicoes,x_tr,y_tr_clf.astype(int),g_tr,
        n_iter=N_ITER_CLASSIFICACAO,scoring="roc_auc",n_splits=5,
    )
    folds,oof=validar_por_grupos(
        clone(busca.best_estimator_),x_tr,y_tr_clf.astype(int),g_tr,n_splits=5
    )
    proba_te=busca.best_estimator_.predict_proba(x_te)[:,1]
    resultados_a[nome]={
        "auc_cv_busca":float(busca.best_score_),
        "auc_oof":float(roc_auc_score(y_tr_clf,oof)),
        "auc_teste":float(roc_auc_score(y_te_clf,proba_te)),
        "parametros":busca.best_params_,
        "modelo":busca.best_estimator_,
        "oof":oof,
        "proba_teste":proba_te,
    }
tab_a=pd.DataFrame({k:{m:v[m] for m in ["auc_cv_busca","auc_oof","auc_teste"]} for k,v in resultados_a.items()}).T
display(tab_a.round(3))


Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits


,auc_cv_busca,auc_oof,auc_teste
logistic_regression,0.835,0.835,0.795
random_forest,0.821,0.826,0.781


## 3. Diagnóstico da mudança temporal do alvo


In [4]:
def auc_univariada(df,coluna,sinal=1):
    dd=df.dropna(subset=[coluna,"atingiu_meta_t"])
    if dd["atingiu_meta_t"].nunique()<2:return np.nan
    return roc_auc_score(dd["atingiu_meta_t"].astype(int),sinal*pd.to_numeric(dd[coluna],errors="coerce"))

candidatas=[
    ("salto_necessario",-1),
    ("indicador_t1",1),
    ("ideb_ideb_observado_ultimo",1),
    ("media_inse",1),
]
linhas=[]
for coluna,sinal in candidatas:
    if coluna in treino.columns:
        linhas.append({
            "variavel":rotulo(coluna),
            f"AUC {ANO_TREINO}":auc_univariada(treino,coluna,sinal),
            f"AUC {ANO_TESTE}":auc_univariada(teste,coluna,sinal),
        })
checagem=pd.DataFrame(linhas).set_index("variavel")
display(checagem.round(3))

deriva_alvo=abs(y_te_clf.astype(float).mean()-y_tr_clf.astype(float).mean())
print(f"Mudança da taxa de atingimento entre os anos: {100*deriva_alvo:.1f} p.p.")


,AUC 2024,AUC 2025
variavel,,
Avanco necessario para atingir a meta (p.p.),0.487,0.748
Alfabetizados no ano anterior (%),0.483,0.646
IDEB mais recente,0.608,0.561
Nivel socioeconomico medio (INSE),0.486,0.408


Mudança da taxa de atingimento entre os anos: 16.9 p.p.


## 4. Formulação B: regressão do nível e risco derivado


In [5]:
CONFIG_REGRESSORES=["ridge","random_forest"]
N_ITER_REGRESSAO=2
resultados_b={}
for nome in CONFIG_REGRESSORES:
    estimador,distribuicoes=regressores(n_jobs=1)[nome]
    pipeline=construir_pipeline(x_tr,estimador)
    busca=buscar_hiperparametros(
        pipeline,distribuicoes,x_tr,y_tr_nivel,g_tr,
        n_iter=N_ITER_REGRESSAO,scoring="neg_mean_absolute_error",n_splits=5,
    )
    folds,oof=validar_regressao_por_grupos(
        clone(busca.best_estimator_),x_tr,y_tr_nivel,g_tr,n_splits=5
    )
    residuos=y_tr_nivel.to_numpy()-oof
    risco_oof=risco_de_nao_atingir(oof,treino.loc[x_tr.index,"meta_t"],residuos)
    pred_te=busca.best_estimator_.predict(x_te)
    risco_te=risco_de_nao_atingir(pred_te,teste.loc[x_te.index,"meta_t"],residuos)
    met_te=avaliar_regressao(y_te_nivel,pred_te)
    resultados_b[nome]={
        "mae_cv_busca":float(-busca.best_score_),
        "mae_teste":met_te["mae"],
        "rmse_teste":met_te["rmse"],
        "r2_teste":met_te["r2"],
        "vies_teste":met_te["vies"],
        "spearman_teste":met_te["spearman"],
        "auc_risco_oof":float(roc_auc_score(y_tr_clf,1-risco_oof)),
        "auc_risco_teste":float(roc_auc_score(y_te_clf,1-risco_te)),
        "parametros":busca.best_params_,
        "modelo":busca.best_estimator_,
        "oof":oof,"pred_teste":pred_te,"risco_oof":risco_oof,"risco_teste":risco_te,
        "residuos":residuos,
    }
metricas_b=["mae_cv_busca","mae_teste","rmse_teste","r2_teste","vies_teste","spearman_teste","auc_risco_oof","auc_risco_teste"]
tab_b=pd.DataFrame({k:{m:v[m] for m in metricas_b} for k,v in resultados_b.items()}).T
display(tab_b.round(3))


Fitting 5 folds for each of 2 candidates, totalling 10 fits
Fitting 5 folds for each of 2 candidates, totalling 10 fits


,mae_cv_busca,mae_teste,rmse_teste,r2_teste,vies_teste,spearman_teste,auc_risco_oof,auc_risco_teste
ridge,7.978,10.746,13.637,0.286,-8.540,0.793,0.823,0.837
random_forest,8.233,9.874,12.444,0.405,-8.053,0.827,0.812,0.864


## 5. Comparação e escolha da formulação


In [7]:
melhor_a=tab_a["auc_teste"].idxmax()
melhor_b=tab_b["auc_risco_teste"].idxmax()
comparacao=pd.DataFrame({
    "classificação direta":{
        "modelo":melhor_a,
        "AUC validação":tab_a.loc[melhor_a,"auc_oof"],
        "AUC teste":tab_a.loc[melhor_a,"auc_teste"],
    },
    "regressão + meta":{
        "modelo":melhor_b,
        "AUC validação":tab_b.loc[melhor_b,"auc_risco_oof"],
        "AUC teste":tab_b.loc[melhor_b,"auc_risco_teste"],
    },
}).T
display(comparacao)

queda_a=comparacao.loc["classificação direta","AUC validação"]-comparacao.loc["classificação direta","AUC teste"]
queda_b=comparacao.loc["regressão + meta","AUC validação"]-comparacao.loc["regressão + meta","AUC teste"]
formulacao_escolhida="regressão + meta" if comparacao.loc["regressão + meta","AUC teste"]>=comparacao.loc["classificação direta","AUC teste"] else "classificação direta"
print(f"Formulação com maior AUC no teste temporal: {formulacao_escolhida}")
print(f"Queda validação→teste: classificação={queda_a:.3f}; regressão+meta={queda_b:.3f}")

resumo={
 "ano_treino":ANO_TREINO,"ano_teste":ANO_TESTE,
 "melhor_classificador":melhor_a,"melhor_regressor":melhor_b,
 "formulacao_escolhida":formulacao_escolhida,
 "classificacao":{k:float(tab_a.loc[melhor_a,k]) for k in tab_a.columns},
 "regressao":{k:float(tab_b.loc[melhor_b,k]) for k in tab_b.columns},
 "features":features,
}

def para_json(valor):
    """Converte tipos NumPy/Pandas para tipos aceitos pelo JSON."""
    if isinstance(valor, dict):
        return {
            str(chave): para_json(item)
            for chave, item in valor.items()
        }

    if isinstance(valor, (list, tuple, set)):
        return [
            para_json(item)
            for item in valor
        ]

    if isinstance(valor, np.ndarray):
        return [
            para_json(item)
            for item in valor.tolist()
        ]

    if isinstance(valor, np.generic):
        return valor.item()

    if isinstance(valor, pd.Timestamp):
        return valor.isoformat()

    if valor is pd.NA:
        return None

    if isinstance(valor, float) and not np.isfinite(valor):
        return None

    return valor


resumo_json = para_json(resumo)

(REPORTS / "comparacao_formulacoes_municipais.json").write_text(
    json.dumps(
        resumo_json,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    ),
    encoding="utf-8",
)

comparacao.to_csv(REPORTS/"comparacao_formulacoes_municipais.csv")
print("Relatórios salvos em reports/.")


,modelo,AUC validação,AUC teste
classificação direta,logistic_regression,0.835289,0.795238
regressão + meta,random_forest,0.812117,0.863693


Formulação com maior AUC no teste temporal: regressão + meta
Queda validação→teste: classificação=0.040; regressão+meta=-0.052
Relatórios salvos em reports/.


## 6. Interpretação

A formulação final não é definida por uma narrativa antecipada. O critério principal é o teste temporal. Se a regressão do nível produzir AUC de risco superior e menor deterioração entre validação e teste, a meta será aplicada somente após a previsão. Se a classificação direta for superior, o resultado deverá ser acompanhado de uma análise explícita sobre a estabilidade da regra administrativa entre os anos.
